In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, TimesFm2_5ModelForPrediction, AutoModel

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, balanced_accuracy_score
import numpy as np
import wandb
wandb.require("core") # Add this line to use the new, stable backend

from utils.load_dataset import load_lsst_data

c:\Users\vujas\Desktop\Etudes\3A\dl_for_timeseries\dl-for-deeplearning-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
wandb: WARNING `wandb.require('core')` is a no-op as it is now the default behavior.


In [12]:
!uv pip show transformers

Name: transformers
Version: 5.3.0
Location: C:\Users\vujas\Desktop\Etudes\3A\dl_for_timeseries\dl-for-deeplearning-project\.venv\Lib\site-packages
Requires: huggingface-hub, numpy, packaging, pyyaml, regex, safetensors, tokenizers, tqdm, typer
Required-by: dl-for-deeplearning-project


Using Python 3.13.9 environment at: C:\Users\vujas\Desktop\Etudes\3A\dl_for_timeseries\dl-for-deeplearning-project\.venv


In [3]:
# Load LSST dataset
X_train, y_train, X_test, y_test = load_lsst_data()

# Encode labels to integers
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)
num_classes = len(label_encoder.classes_)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_encoded, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_encoded, dtype=torch.long)

# Create DataLoaders
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [4]:
class TimesFMClassifier(nn.Module):
    def __init__(self, model_name="google/timesfm-2.5-200m-transformers", num_classes=14):
        super().__init__()
        
        # Load the base model without the forecasting head
        self.foundation_model = TimesFm2_5ModelForPrediction.from_pretrained(model_name)
        
        # Linear Probing: Freeze the base model
        for param in self.foundation_model.parameters():
            param.requires_grad = False
            
        # Extract the hidden dimension size
        hidden_size = self.foundation_model.config.hidden_size
        
        # Define the linear classification head
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # x shape: (batch_size, n_channels, n_timesteps)
        batch_size, n_channels, n_timesteps = x.shape
        
        # Reshape to process all channels independently: (batch_size * n_channels, n_timesteps)
        x_reshaped = x.view(batch_size * n_channels, n_timesteps)
        
        outputs = self.foundation_model(past_values=x_reshaped, output_hidden_states=True)
        
        # Get hidden states and pool them over the time dimension
        hidden_states = outputs.last_hidden_state # (B*C, SeqLen, Hidden)
        pooled_embeddings = hidden_states.mean(dim=1) # (B*C, Hidden)
        
        # Reshape back to (batch_size, n_channels, hidden_size) and average across the 6 channels
        pooled_embeddings = pooled_embeddings.view(batch_size, n_channels, -1)
        final_embedding = pooled_embeddings.mean(dim=1) # (B, Hidden)
        
        # Pass the final embedding through the linear layer
        logits = self.classifier(final_embedding)
        return logits

model = TimesFMClassifier(num_classes=num_classes)

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 1115.94it/s]


In [5]:
print(model)

TimesFMClassifier(
  (foundation_model): TimesFm2_5ModelForPrediction(
    (model): TimesFm2_5Model(
      (input_ff_layer): TimesFm2_5ResidualBlock(
        (input_layer): Linear(in_features=64, out_features=1280, bias=True)
        (activation): SiLU()
        (output_layer): Linear(in_features=1280, out_features=1280, bias=True)
        (residual_layer): Linear(in_features=64, out_features=1280, bias=True)
      )
      (layers): ModuleList(
        (0-19): 20 x TimesFm2_5DecoderLayer(
          (self_attn): TimesFm2_5Attention(
            (q_proj): Linear(in_features=1280, out_features=1280, bias=False)
            (k_proj): Linear(in_features=1280, out_features=1280, bias=False)
            (v_proj): Linear(in_features=1280, out_features=1280, bias=False)
            (o_proj): Linear(in_features=1280, out_features=1280, bias=False)
            (q_norm): TimesFm2_5RMSNorm((80,), eps=1e-06)
            (k_norm): TimesFm2_5RMSNorm((80,), eps=1e-06)
          )
          (mlp): Times

In [6]:
# config = {
#         "learning_rate": 1e-3,
#         "epochs": 10,
#         "batch_size": 32,
#         "architecture": "TimesFM-Adapter",
#         "dataset": "LSST"
#     }

In [7]:
# Initialize a new run in wandb
wandb.init(
    entity="laveinev2-t-l-com-paris",
    project="dl-for-time-series",
    name="timesfm-linear-probing",
    mode = "offline",
    config={
       
        "learning_rate": 1e-3,
        "epochs": 10,
        "batch_size": 32,
        "architecture": "TimesFM-Adapter",
        "dataset": "LSST"
    }
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.CrossEntropyLoss()
# Pass only the newly initialized classifier parameters to the optimizer
optimizer = optim.Adam(model.classifier.parameters(), lr=wandb.config.learning_rate)

for epoch in range(wandb.config.epochs):
    # --- Training Phase ---
    model.train()
    total_loss = 0
    
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    avg_train_loss = total_loss / len(train_loader)
    
    # --- Evaluation Phase ---
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X = batch_X.to(device)
            outputs = model(batch_X)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            
            all_preds.extend(preds)
            all_targets.extend(batch_y.numpy())
            
    # Calculate robust metrics for imbalanced classes
    macro_f1 = f1_score(all_targets, all_preds, average="macro")
    bal_acc = balanced_accuracy_score(all_targets, all_preds)
    
    print(f"Epoch {epoch+1}/{wandb.config.epochs} | Loss: {avg_train_loss:.4f} | Macro F1: {macro_f1:.4f} | Bal Acc: {bal_acc:.4f}")
    
    # Push metrics to the Weights & Biases dashboard
    wandb.log({
        "train_loss": avg_train_loss,
        "val_macro_f1": macro_f1,
        "val_balanced_accuracy": bal_acc,
        "epoch": epoch + 1
    })

# Close the wandb run to finalize the logs
wandb.finish()

ServicePollForTokenError: Failed to read port info after 30.0 seconds.